In [1]:
# Load environment and suppress warnings

import os
import warnings
from dotenv import load_dotenv
                
load_dotenv()
warnings.filterwarnings("ignore")

api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

Key loaded: True


In [2]:
# Import components and initialize deterministic model

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

model = ChatOpenAI(model="gpt-5-mini", temperature=0)

In [3]:
# Define strict schema where priority must be exactly Low, Medium, or High

from enum import Enum
from pydantic import BaseModel

class Priority(str, Enum):
    LOW = "Low"
    MEDIUM = "Medium"
    HIGH = "High"

class Ticket(BaseModel):
    priority: Priority
    issue: str
    action: str

parser = PydanticOutputParser(pydantic_object=Ticket)
format_instructions = parser.get_format_instructions()

In [4]:
# Use a simple ticket input that we will classify

ticket_text = """
Cosmetic alignment issue in footer.
No user impact.
Suggested action: add to backlog.
""".strip()

print(ticket_text)

Cosmetic alignment issue in footer.
No user impact.
Suggested action: add to backlog.


In [5]:
# Generate a raw response that intentionally violates the enum by using P1/P2/P3

raw_prompt = PromptTemplate(
    input_variables=["ticket_text"],
    template=(
        "Extract fields as JSON with keys priority, issue, action.\n"
        "Priority must be P1, P2, or P3.\n"
        "Return only JSON.\n\n"
        "Ticket:\n{ticket_text}\n"
    ),
)

raw_chain = raw_prompt | model | StrOutputParser()
raw_output = raw_chain.invoke({"ticket_text": ticket_text})

print("Raw model output:")
print(raw_output)

Raw model output:
{"priority":"P3","issue":"Cosmetic alignment issue in footer; no user impact.","action":"Add to backlog."}


In [6]:
# Attempt to parse the raw output using the strict schema and show the validation failure

try:
    parsed = parser.parse(raw_output)
    print("Unexpected success:", parsed)
except Exception as e:
    print("Validation error triggered (expected):")
    print(e)

Validation error triggered (expected):
Failed to parse Ticket from completion {"priority": "P3", "issue": "Cosmetic alignment issue in footer; no user impact.", "action": "Add to backlog."}. Got: 1 validation error for Ticket
priority
  Input should be 'Low', 'Medium' or 'High' [type=enum, input_value='P3', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/enum
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 


In [7]:
# Retry by asking the model to reformat using the allowed enum values and schema instructions

retry_prompt = PromptTemplate(
    input_variables=["ticket_text"],
    template=(
        "Extract structured data from the ticket.\n"
        "Priority MUST be exactly one of: Low, Medium, High.\n"
        "Return output strictly in the required schema.\n"
        "Do not add explanations.\n\n"
        "{format_instructions}\n\n"
        "Ticket:\n{ticket_text}\n"
    ),
).partial(format_instructions=format_instructions)

retry_chain = retry_prompt | model | parser
result = retry_chain.invoke({"ticket_text": ticket_text})

print("Recovered result:")
print(result)

Recovered result:
priority=<Priority.LOW: 'Low'> issue='Cosmetic alignment issue in footer' action='Add to backlog'


In [8]:
# Demonstrate fallback by intentionally forcing retry to fail

def safe_extract_with_forced_failure(ticket_text):
    try:
        # Intentionally parse raw invalid output again to force failure
        parser.parse(raw_output)
        return retry_chain.invoke({"ticket_text": ticket_text})
    except Exception:
        print("Retry failed. Applying fallback.")
        return Ticket(
            priority=Priority.LOW,
            issue="Unable to validate model output",
            action="Send for manual review"
        )

fallback_result = safe_extract_with_forced_failure(ticket_text)
print("Fallback output:")
print(fallback_result.model_dump())

Retry failed. Applying fallback.
Fallback output:
{'priority': <Priority.LOW: 'Low'>, 'issue': 'Unable to validate model output', 'action': 'Send for manual review'}
